# fastText

https://github.com/facebookresearch/fastText

**FastText**는 자연어 처리(NLP) 작업에서 사용되는 오픈소스 라이브러리로, 텍스트 분류 및 단어 임베딩을 위한 빠르고 효율적인 도구이다. 이는 Facebook AI Research 팀에서 개발했으며, 특히 대규모 텍스트 데이터에서도 높은 성능과 속도를 제공한다.

FastText는 2016년에 Facebook AI Research(FAIR) 팀에 의해 처음 공개되었다.  
처음 발표된 논문은 **"Enriching Word Vectors with Subword Information"** (2016)이며, 이후 오픈소스로 GitHub에 공개되어 많은 사용자들에게 활용되고 있다.  

FastText는 Word2Vec 이후 등장한 모델로, 특히 서브워드 정보 활용과 텍스트 분류에서 효율성을 강조하면서 주목받았다.

**주요 특징**
1. **단어 벡터 학습 (Word Embeddings)**  
   - FastText는 단어를 고정된 크기의 벡터로 변환하는 단어 임베딩 모델을 학습한다. 이는 단어의 의미를 벡터 공간에 매핑하여 유사한 단어가 가까운 벡터로 표현되도록 한다.
   - 기존의 Word2Vec과 유사하지만, FastText는 단어를 **서브워드(subword)** 단위로 처리한다.

2. **서브워드 기반 모델 (Subword-based Model)**  
   - 단어를 n-그램(예: 'apple' → ['app', 'ppl', 'ple'])으로 분해하여 학습하기 때문에, **희귀 단어**나 **철자 오류**에도 강건하다.
   - 이는 단어 외에도 철자 패턴과 같은 더 세밀한 정보를 학습하는 데 유용하다.

3. **텍스트 분류 (Text Classification)**  
   - FastText는 문서나 문장을 빠르고 정확하게 분류하는 데 최적화되어 있다.
   - 학습 과정이 빠르고, 모델의 크기가 작으며, 정확도도 뛰어나다.

4. **효율적인 구현**  
   - FastText는 CPU 기반으로도 높은 성능을 내도록 설계되었으며, 대규모 데이터셋에서도 빠르게 작동한다.

**FastText의 작동 원리**
1. **단어 표현**  
   - 단어를 n-그램 서브워드로 나눈 후, 각 서브워드에 대해 벡터를 학습한다.
   - 예를 들어, "cat"이라는 단어는 'c', 'ca', 'cat'과 같은 다양한 조합으로 분해된다.
   - 결과적으로 단어 벡터는 각 서브워드 벡터의 합으로 표현된다.

2. **모델 구조**  
   - FastText는 Skip-gram 모델이나 CBOW 모델을 기반으로 동작한다.
   - 단, 기존 모델과 달리 단어 자체가 아닌 서브워드를 사용하여 학습한다.

**FastText의 장점**
1. **희귀 단어 처리 능력**  
   - 서브워드 기반 접근 방식 덕분에 희귀 단어 또는 새로운 단어에 대해 더 좋은 일반화 성능을 발휘한다.
2. **빠른 학습 속도**  
   - 단순한 모델 구조와 최적화된 구현으로 매우 빠르게 학습할 수 있다.
3. **다양한 언어 지원**  
   - 다양한 언어에서 동작하며, 특히 굴절어(inflected languages)와 같은 복잡한 언어에서도 효과적이다.

**활용 사례**
1. **단어 임베딩**  
   - 단어 간 유사도 계산, 문장 표현 학습.
2. **텍스트 분류**  
   - 스팸 필터링, 감정 분석, 뉴스 분류.
3. **다언어 지원**  
   - 다국어 데이터셋에서 빠른 응답 성능 제공.

## gensim - fastText

In [1]:
from lxml import etree

xml = open('ted_en.xml', 'r', encoding='UTF8')
tree = etree.parse(xml)

content = tree.xpath('//content/text()')
print(len(content))

corpus = '\n'.join(content)

2085


In [2]:
# 정제
import re
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

corpus = re.sub(r'\([^)]*\)', '', corpus)
corpus = re.sub(r'[^a-z0-9\s.,!?]+', '', corpus, flags=re.IGNORECASE)
corpus = corpus.lower()

en_stopwords = stopwords.words('english')

sentences = sent_tokenize(corpus)
normalized_setences = []

for sent in sentences:
    tokens = word_tokenize(sent)
    normalized_tokens = [token for token in tokens if token not in en_stopwords]
    normalized_setences.append(normalized_tokens)

print('총 문장수: ', len(normalized_setences))

총 문장수:  271088


### Word2Vec/FastText 임베딩 모델 비교

In [3]:
from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentences=normalized_setences,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    sg=1
)

w2v_model.wv.vectors.shape

(22174, 100)

In [4]:
from gensim.models.fasttext import FastText

ft_model = FastText(
    sentences=normalized_setences,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    sg=1
)

ft_model.wv.vectors.shape

(22174, 100)

In [5]:
# 단어 사전 확인
print(list(w2v_model.wv.key_to_index.keys())[:10])
print(list(ft_model.wv.key_to_index.keys())[:10])

[',', '.', '?', 'one', 'people', 'like', 'know', 'going', 'think', 'thats']
[',', '.', '?', 'one', 'people', 'like', 'know', 'going', 'think', 'thats']


In [6]:
# 학습 된 벡터 값 확인
print(w2v_model.wv['computer'][:10])
print(ft_model.wv['computer'][:10])

[ 0.10528731 -0.02765989 -0.0321636   0.04940296 -0.40209046 -0.63934594
  0.18604378  0.39935678  0.20105311 -0.42474204]
[-0.10319624  0.14984378  0.00924194  0.218552   -0.03797586 -0.9599098
 -0.6177751   0.08515587  0.09967085 -0.57606006]


In [10]:
# OOV 데이터에 대한 조회

# KeyError: "Key 'electrofishing' not present in vocabulary"
# print(w2v_model.wv.most_similar('electrofishing'))
print(ft_model.wv.most_similar('electrofishing'))

# 해당 단어는 wv에 존재하지 않지만 유사도 기반으로 검색이 가능하다.
# KeyError: "Key 'electrofishing' not present"
# print(ft_model.wv.get_index('electrofishing'))

[('electrolux', 0.8657050132751465), ('hydroelectric', 0.8640168905258179), ('electrolyte', 0.8541896939277649), ('electroshock', 0.8457106947898865), ('electric', 0.8317040205001831), ('solarelectrified', 0.8315333127975464), ('electronic', 0.8279218077659607), ('electroencephalogram', 0.8267300128936768), ('electrons', 0.8257879018783569), ('electrochemical', 0.8256867527961731)]


In [14]:
import numpy as np

word = 'apple'

# 단어의 사전 index 확인
word_idx = ft_model.wv.get_index(word)

# 모델이 실제로 반환하는 최종 단어 벡터
word_vec = ft_model.wv[word]
print(word_vec[:10])

# 단어 자체에 대한 기본 벡터
base_vec = ft_model.wv.vectors_vocab[word_idx]
print(base_vec[:10])

# 해당 단어를 이루는 subword들의 bucket index 확인
subword_idx = ft_model.wv.buckets_word[word_idx]

# subword 벡터들 조회
subword_vecs = ft_model.wv.vectors_ngrams[subword_idx]
print(subword_vecs.shape)

# 단어 자체 벡터와 subword 벡터를 함께 쌓은 뒤 평균낸 결과 값 출력
cal_vec = np.vstack((base_vec, subword_vecs)).mean(axis=0)
print(cal_vec[:10])


[-0.08853292  0.91159785  0.07231873  0.31952745 -0.08001427 -0.9655049
 -0.24855863  0.25873417  0.1014619  -0.73489976]
[-0.346858    0.04860941  0.08223782  0.18085174  0.18955311 -0.3362674
  0.12060849 -0.25417852  0.13171527 -0.35292518]
(14, 100)
[-0.08853292  0.91159785  0.07231873  0.31952745 -0.08001427 -0.9655049
 -0.24855863  0.25873417  0.1014619  -0.73489976]


In [15]:
# 문장 벡터
sent_vec = ft_model.wv.get_sentence_vector("i love apple very much")
sent_vec.shape

(100,)

In [18]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

sentences = [
    "i love apple",
    "apple is good",
    "i hate banana",
    "banana is yellow"
]

sent_vecs = np.array([ft_model.wv.get_sentence_vector(sent) for sent in sentences])
print(sent_vecs.shape)

# 문장 유사도 확인
sent_cos_sim = cosine_similarity(sent_vecs)

pd.DataFrame(sent_cos_sim, columns=sentences, index=sentences)


(4, 100)


,i love apple,apple is good,i hate banana,banana is yellow
i love apple,1.000000,0.930130,0.781537,0.909072
apple is good,0.930130,1.000000,0.724401,0.866048
i hate banana,0.781537,0.724401,1.000000,0.902531
banana is yellow,0.909072,0.866048,0.902531,1.000000


In [19]:
# 모델 저장
ft_model.save('ted_en_fasttext.model')

In [20]:
# 모델 로드
ft_model2 = FastText.load('ted_en_fasttext.model')
ft_model2.wv.most_similar('computer')

[('compute', 0.9330584406852722),
 ('supercomputer', 0.9167945981025696),
 ('computers', 0.8992694020271301),
 ('humancomputer', 0.8757193684577942),
 ('computerized', 0.864450991153717),
 ('supercomputers', 0.8273196220397949),
 ('computergenerated', 0.8255923986434937),
 ('compulsory', 0.7861198782920837),
 ('computing', 0.7859460115432739),
 ('programmer', 0.7639650702476501)]